In [40]:
#import the required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

In [41]:
#load the cleaned dataset

df = pd.read_csv('cleaned_materials.csv')
df.head()

,Material_ID,Strength_MPa,Weight_Capacity_kg,Biodegradability_Score,CO2_Emission_Score_kg,Recyclability_Percentage,Source,Certification,Price_INR,Material_Type_Bio-based Polymer,...,Product_Category_Kitchen Utensils,Product_Category_Office Supplies,Product_Category_Sports Equipment,Product_Category_Textiles,Product_Category_Toys,Availability_Status_Low Stock,Availability_Status_Pre-Order,CO2_Impact_Index,Cost_Efficiency_Index,Material_Suitability_Score
0,MAT00001,0.024921,0.275033,0.22375,0.736552,0.677,Supplier A,EcoCert,0.892455,False,...,False,False,False,True,False,True,False,0.575854,0.357736,0.280193
1,MAT00002,0.093601,0.232596,0.60250,0.561379,0.716,Supplier D,FSC,0.701542,False,...,False,False,False,False,False,True,False,0.640459,0.420795,0.432991
2,MAT00003,0.809409,0.006366,0.80625,0.697931,0.340,Supplier C,ISO 14001,0.155481,False,...,False,False,False,False,False,False,False,0.588952,0.294250,0.667639
3,MAT00004,0.358914,0.343943,0.26500,0.043448,0.459,Supplier D,ISO 14001,0.124844,False,...,False,False,False,True,False,False,True,0.958361,0.408057,0.360766
4,MAT00005,0.885437,0.361625,0.19250,0.069655,0.661,Supplier A,FSC,0.773447,False,...,False,False,False,False,False,False,False,0.934881,0.372720,0.610225


In [42]:
#Define Features (X) & Targets (Y)

X = df.drop(columns=["Price_INR", "CO2_Emission_Score_kg"])
y_cost = df["Price_INR"]
y_co2 = df["CO2_Emission_Score_kg"]


In [43]:
#Keep only numerical columns

X = X.drop(columns=[
    "Material_ID",
    "Source",
    "Certification"
], errors="ignore")


In [44]:
#Train - Test split

X_train, X_test, y_cost_train, y_cost_test = train_test_split(X, y_cost, test_size=0.2, random_state=42)

In [45]:
#Train the random forest regressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_cost_train)


RandomForestRegressor(random_state=42)

In [46]:
#Evaluate random forest model

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

cost_pred = rf_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_cost_test, cost_pred))
mae = mean_absolute_error(y_cost_test, cost_pred)
r2 = r2_score(y_cost_test, cost_pred)

print("Random Forest – Cost Prediction Performance")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2 Score:", r2)


Random Forest – Cost Prediction Performance
RMSE: 0.03425175348195902
MAE: 0.01396171112963736
R2 Score: 0.9861053714933236


In [47]:
#Train XGBoost Regressor

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_co2_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [48]:
#Evaluate XGBoost model

co2_pred = xgb_model.predict(X_test)

rmse_co2 = np.sqrt(mean_squared_error(y_co2_test, co2_pred))
mae_co2 = mean_absolute_error(y_co2_test, co2_pred)
r2_co2 = r2_score(y_co2_test, co2_pred)

print("XGBoost – CO₂ Prediction")
print("RMSE:", rmse_co2)
print("MAE:", mae_co2)
print("R2 Score:", r2_co2)


XGBoost – CO₂ Prediction
RMSE: 0.0011850575670215666
MAE: 0.0009965261434074836
R2 Score: 0.9999825089036263


In [49]:
#Generate predictions for the entire dataset

df["Predicted_Cost"] = rf_model.predict(X)
df["Predicted_CO2"] = xgb_model.predict(X)


In [50]:
#Normalize the predicted values

from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()

df[["Predicted_Cost_Norm", "Predicted_CO2_Norm"]] = scaler.fit_transform(
    df[["Predicted_Cost", "Predicted_CO2"]]
)


In [51]:
#Calculate final ranking score

df["Final_Ranking_Score"] = (
    0.5 * (1 - df["Predicted_Cost_Norm"]) +
    0.5 * (1 - df["Predicted_CO2_Norm"])
)


In [52]:
#Create reatdable Material_Type_Label

material_cols = [col for col in df.columns if col.startswith("Material_Type_")]

df["Material_Type_Label"] = (
    df[material_cols]
    .idxmax(axis=1)
    .str.replace("Material_Type_", "")
)


In [53]:
#Display top 10 recommended materials

df_ranked = df.sort_values("Final_Ranking_Score", ascending=False)

df_ranked[
    [
        "Material_Type_Label",
        "Predicted_Cost",
        "Predicted_CO2",
        "Final_Ranking_Score"
    ]
].head(10)


,Material_Type_Label,Predicted_Cost,Predicted_CO2,Final_Ranking_Score
3854,Natural Rubber,0.008114,0.009942,0.994626
4733,Certified Wood,0.016478,0.014700,0.988004
7563,Seaweed Polymer,0.029915,0.006661,0.985228
706,Sustainable Bamboo,0.018762,0.018250,0.985067
9837,Wool,0.032270,0.010081,0.982320
7602,Bio-based Polymer,0.019218,0.024433,0.981735
6359,Recycled Aluminum,0.021251,0.027830,0.979002
9318,Bio-based Polymer,0.049596,0.005613,0.975784
2960,Natural Rubber,0.054325,0.001590,0.975406
9661,Bio-based Polymer,0.019456,0.037785,0.974920
